<a href="https://colab.research.google.com/github/blakdestiny/The-Market-Gap-Analysis/blob/main/Sugar_Trap_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import urllib.request

# Download the gzipped export (0.9GB — takes a couple minutes on Colab)
url = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
urllib.request.urlretrieve(url, "products.csv.gz")


('products.csv.gz', <http.client.HTTPMessage at 0x7bef7ae79b20>)

In [17]:
# Only pull the columns you actually need — this is what makes 12GB manageable
cols = [
    "product_name", "categories_tags", "ingredients_text",
    "sugars_100g", "proteins_100g", "fat_100g",
    "nutriscore_grade", "countries_tags"
]

snack_keywords = "snack|biscuit|cookie|chip|cracker|bar|chocolate|cereal|candy"

chunks = []
reader = pd.read_csv(
    "products.csv.gz",
    sep="\t",
    compression="gzip",
    usecols=cols,
    chunksize=100_000,
    low_memory=False,
    on_bad_lines="skip",
)

for chunk in reader:
    mask = chunk["categories_tags"].str.contains(snack_keywords, case=False, na=False)
    filtered = chunk[mask]
    if not filtered.empty:
        chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

(521231, 8)


In [18]:
print(f"Starting rows: {len(df)}")

# --- Step 1: Drop rows missing the fields we can't analyze without ---
required = ["product_name", "sugars_100g", "proteins_100g"]
df_clean = df.dropna(subset=required).copy()

# Also drop empty-string product names (nulls aren't the only "missing")
df_clean = df_clean[df_clean["product_name"].str.strip() != ""]

print(f"After dropping missing required fields: {len(df_clean)}")

Starting rows: 521231
After dropping missing required fields: 321586


In [19]:
# --- Step 2: Bound-check biologically impossible values ---
# Per-100g fields can't exceed 100 (you can't have more than 100g
# of a nutrient in 100g of food). fat_100g can occasionally read
# close to 100 for pure oils, so we leave a little headroom there.
def in_range(series, low=0, high=100):
    return series.between(low, high)

before = len(df_clean)
df_clean = df_clean[
    in_range(df_clean["sugars_100g"]) &
    in_range(df_clean["proteins_100g"]) &
    in_range(df_clean["fat_100g"].fillna(0), high=100)
]
print(f"After removing out-of-range nutrients: {len(df_clean)} (-{before - len(df_clean)})")

TypeError: '>=' not supported between instances of 'str' and 'int'

In [20]:
print(df_clean[["sugars_100g", "proteins_100g", "fat_100g"]].dtypes)

sugars_100g      float64
proteins_100g    float64
fat_100g          object
dtype: object


In [21]:
cols = ["sugars_100g", "proteins_100g", "fat_100g"]

for col in cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

In [22]:
def in_range(series, low=0, high=100):
    return series.between(low, high)

before = len(df_clean)

df_clean = df_clean[
    in_range(df_clean["sugars_100g"]) &
    in_range(df_clean["proteins_100g"]) &
    in_range(df_clean["fat_100g"].fillna(0))
]

print(f"After removing out-of-range nutrients: {len(df_clean)} (-{before - len(df_clean)})")

After removing out-of-range nutrients: 321461 (-125)


In [23]:
# --- Step 3: Drop exact duplicate products (same name + same macros) ---
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["product_name", "sugars_100g", "proteins_100g"])
print(f"After deduping: {len(df_clean)} (-{before - len(df_clean)})")

After deduping: 301010 (-20451)


In [24]:
# --- Step 4: Sanity summary — worth screenshotting for your README's Technical Explanation section ---
print(df_clean[["sugars_100g", "proteins_100g", "fat_100g"]].describe())

         sugars_100g  proteins_100g       fat_100g
count  301010.000000  301010.000000  300683.000000
mean       20.803746       7.927090      15.720374
std        20.833993       5.801409      13.765661
min         0.000000       0.000000       0.000000
25%         2.500000       5.000000       3.000000
50%        15.000000       7.000000      13.500000
75%        34.000000       9.700000      25.000000
max       100.000000     100.000000     100.000000


In [25]:
# --- Category Wrangler: map messy categories_tags into readable buckets ---

# Order matters — first match wins, so put more specific categories
# before general ones (e.g. "Bars" before "Chocolate/Candy" so a
# chocolate protein bar doesn't get miscategorized).
category_rules = [
    ("Bars",               ["cereal-bar", "protein-bar", "granola-bar", "energy-bar"]),
    ("Chips/Crisps",       ["chips", "crisps", "tortilla-chip"]),
    ("Crackers",           ["cracker"]),
    ("Cookies/Biscuits",   ["biscuit", "cookie"]),
    ("Chocolate/Candy",    ["chocolate", "candy", "confectionery", "sweet"]),
    ("Nuts/Seeds",         ["nuts", "seeds", "trail-mix"]),
    ("Cereal/Granola",     ["cereal", "granola", "muesli"]),
]

def assign_category(tags):
    if pd.isna(tags):
        return "Other"
    tags_lower = tags.lower()
    for category, keywords in category_rules:
        if any(kw in tags_lower for kw in keywords):
            return category
    return "Other"

df_clean["primary_category"] = df_clean["categories_tags"].apply(assign_category)

In [26]:
# Sanity check: how did the buckets fill out?
print(df_clean["primary_category"].value_counts())

primary_category
Chocolate/Candy     93961
Cereal/Granola      86014
Cookies/Biscuits    45926
Crackers            19978
Other               19198
Chips/Crisps        15259
Nuts/Seeds          13180
Bars                 7494
Name: count, dtype: int64


In [27]:
df_clean[df_clean["primary_category"] == "Other"]["categories_tags"].value_counts().head(20)

,count
categories_tags,
en:snacks,2894
"en:snacks,en:salty-snacks",1698
"en:snacks,en:salty-snacks,en:appetizers",1225
"en:condiments,en:sauces,en:barbecue-sauces,en:groceries",792
"en:meats-and-their-products,en:meats,en:prepared-meats,en:sausages,en:french-sausages,en:chipolatas",776
"en:snacks,en:popcorn",699
"en:condiments,en:sauces,en:barbecue-sauces",545
"en:desserts,en:frozen-foods,en:frozen-desserts,en:ice-creams-and-sorbets,en:ice-creams,en:ice-cream-bars",486
"en:snacks,en:salty-snacks,en:popcorn,en:salted-popcorn",291


In [1]:
import plotly.express as px

fig = px.scatter(
    df_clean,
    x="sugars_100g",
    y="proteins_100g",
    color="primary_category",
    hover_data=["product_name"],
    opacity=0.6,
    title="Sugar vs Protein by Category (per 100g)",
)
fig.update_layout(
    xaxis_title="Sugar (g / 100g)",
    yaxis_title="Protein (g / 100g)",
)
fig.show()

NameError: name 'df_clean' is not defined

In [3]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import plotly.express as px

df = pd.read_csv("snacks_clean.csv")

st.title("Sugar Trap: Snack Market Gap Analysis")

categories = st.multiselect(
    "Filter by category",
    options=df["primary_category"].unique(),
    default=df["primary_category"].unique(),
)
filtered = df[df["primary_category"].isin(categories)]

fig = px.scatter(filtered, x="sugars_100g", y="proteins_100g", color="primary_category", opacity=0.6)
st.plotly_chart(fig, use_container_width=True)

st.subheader("Key Insight")
st.info("Based on the data, the biggest market opportunity is in [Category], "
        "specifically targeting products with [X]g of protein and less than [Y]g of sugar.")

Writing streamlit_app.py


In [6]:
print(gap_table.sort_values("hp_ls_share"))

NameError: name 'gap_table' is not defined

In [7]:
print("gap_table" in globals())

False


In [8]:
print(dfclean.shape if 'df_clean' in dir() else "df_clean missing")
print("quadrant" in df_clean.columns if 'df_clean' in dir() else "n/a")


df_clean missing
n/a


In [9]:
import pandas as pd
import urllib.request

url = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
urllib.request.urlretrieve(url, "products.csv.gz")

('products.csv.gz', <http.client.HTTPMessage at 0x7c629bba15b0>)

In [10]:
cols = [
    "product_name", "categories_tags", "ingredients_text",
    "sugars_100g", "proteins_100g", "fat_100g",
    "nutriscore_grade", "countries_tags"
]

snack_keywords = "snack|biscuit|cookie|chip|cracker|bar|chocolate|cereal|candy"

chunks = []
reader = pd.read_csv(
    "products.csv.gz", sep="\t", compression="gzip",
    usecols=cols, chunksize=100_000, low_memory=False, on_bad_lines="skip",
)

for chunk in reader:
    mask = chunk["categories_tags"].str.contains(snack_keywords, case=False, na=False)
    filtered = chunk[mask]
    if not filtered.empty:
        chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

(521231, 8)


In [11]:
required = ["product_name", "sugars_100g", "proteins_100g"]
df_clean = df.dropna(subset=required).copy()
df_clean = df_clean[df_clean["product_name"].str.strip() != ""]

numeric_cols = ["sugars_100g", "proteins_100g", "fat_100g"]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean = df_clean.dropna(subset=["sugars_100g", "proteins_100g"])

def in_range(series, low=0, high=100):
    return series.between(low, high)

df_clean = df_clean[
    in_range(df_clean["sugars_100g"]) &
    in_range(df_clean["proteins_100g"]) &
    in_range(df_clean["fat_100g"].fillna(0), high=100)
]

df_clean = df_clean.drop_duplicates(subset=["product_name", "sugars_100g", "proteins_100g"])
print(f"Cleaned rows: {len(df_clean)}")

Cleaned rows: 301010


In [12]:
category_rules = [
    ("Bars",               ["cereal-bar", "protein-bar", "granola-bar", "energy-bar"]),
    ("Chips/Crisps",       ["chips", "crisps", "tortilla-chip"]),
    ("Crackers",           ["cracker"]),
    ("Cookies/Biscuits",   ["biscuit", "cookie"]),
    ("Chocolate/Candy",    ["chocolate", "candy", "confectionery", "sweet"]),
    ("Nuts/Seeds",         ["nuts", "seeds", "trail-mix"]),
    ("Cereal/Granola",     ["cereal", "granola", "muesli"]),
]

def assign_category(tags):
    if pd.isna(tags):
        return "Other"
    tags_lower = tags.lower()
    for category, keywords in category_rules:
        if any(kw in tags_lower for kw in keywords):
            return category
    return "Other"

df_clean["primary_category"] = df_clean["categories_tags"].apply(assign_category)
print(df_clean["primary_category"].value_counts())

primary_category
Chocolate/Candy     93961
Cereal/Granola      86014
Cookies/Biscuits    45926
Crackers            19978
Other               19198
Chips/Crisps        15259
Nuts/Seeds          13180
Bars                 7494
Name: count, dtype: int64
